# NumCompute Stream: Assignment 2.2 Demo

This notebook demonstrates the full streaming workflow I built for COMP 5004 Programming Task 2.

**What you will see:**
* Custom CSV loading through `io.py`
* Chunk by chunk `partial_fit` training on wine data
* `StreamTrainer` logging accuracy and memory after each chunk
* A pipeline with `Imputer`, `StandardScaler`, and tree based ensembles
* Required plots: `plot_metric_over_time`, `compare_models`, and `plot_predictions_vs_ground_truth`

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from numcompute_stream.io import load_csv, split_features_labels, stream_chunks
from numcompute_stream.preprocessing import StandardScaler, Imputer
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.ensemble import BaggingClassifier, RandomForestClassifier
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.stats import update_stats
from numcompute_stream.visualise import (
    plot_metric_over_time,
    compare_models,
    plot_predictions_vs_ground_truth,
    plot_confusion_matrix,
)

DATA = Path("data/wine_stream.csv")
raw = load_csv(DATA)
X, y = split_features_labels(raw, label_col=-1)
classes = np.unique(y)
print(f"Loaded {X.shape[0]} samples, {X.shape[1]} features, classes={classes}")

In [ ]:
# Streaming statistics (stats.py update_stats API)
stats = None
for X_chunk, _ in stream_chunks(X, y, chunk_size=8):
    stats = update_stats(X_chunk, stats)
print("Running mean (first 4 features):", np.round(stats.mean()[:4], 3))

In [ ]:
def make_pipeline(model):
    return Pipeline([
        ("impute", Imputer()),
        ("scale", StandardScaler()),
        ("model", model),
    ])

trainers = {
    "DecisionTree": StreamTrainer(
        make_pipeline(DecisionTreeClassifier(max_depth=5, criterion="gini", random_state=0)),
        classes=classes,
    ),
    "Bagging": StreamTrainer(
        make_pipeline(BaggingClassifier(n_estimators=5, max_depth=4, random_state=0)),
        classes=classes,
    ),
    "RandomForest": StreamTrainer(
        make_pipeline(RandomForestClassifier(n_estimators=5, max_depth=4, random_state=0)),
        classes=classes,
    ),
}

last_X, last_y = None, None
for step, (X_chunk, y_chunk) in enumerate(stream_chunks(X, y, chunk_size=8)):
    last_X, last_y = X_chunk, y_chunk
    for name, trainer in trainers.items():
        log = trainer.fit_chunk(X_chunk, y_chunk)
    print(f"Chunk {step + 1} cumulative accuracy:", {
        n: round(trainers[n].chunk_logs[-1]["cumulative_accuracy"], 3) for n in trainers
    })

In [ ]:
# Spec: plot_metric_over_time
rf_curve = [log["cumulative_accuracy"] for log in trainers["RandomForest"].chunk_logs]
plot_metric_over_time(rf_curve, "Random Forest cumulative accuracy", "Accuracy")
plt.show()

In [ ]:
# Spec: compare_models
tree_curve = [log["cumulative_accuracy"] for log in trainers["DecisionTree"].chunk_logs]
compare_models(tree_curve, rf_curve, ("DecisionTree", "RandomForest"), ylabel="Accuracy")
plt.show()

In [ ]:
# Spec: plot_predictions_vs_ground_truth (latest chunk)
best = max(trainers, key=lambda n: trainers[n].results()["accuracy"])
pred_last = trainers[best].pipeline.predict(last_X)
plot_predictions_vs_ground_truth(last_y, pred_last, title=f"Latest chunk: {best}")
plt.show()

r = trainers[best].results()
print(f"Best model: {best}")
print(f"Accuracy={r['accuracy']:.2%} Precision={r['precision']:.2%} Recall={r['recall']:.2%} F1={r['f1']:.2%}")
plot_confusion_matrix(r["confusion_matrix"], labels=[str(c) for c in classes], title=f"Confusion matrix: {best}")
plt.show()